In [1]:
# import libraries
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import joblib
import os

In [2]:
# 1. Load dataset
df = pd.read_csv('../data/processed_data/full_wildfire_weather_2020_2024.csv')

In [3]:
# 2. Define predictors and targets
predictors = ['startdateseason', 'u10', 'v10', 'd2m', 't2m', 'msl', 'sp', 'lai_hv', 'lai_lv', 'tp', 'ssr']
target1 = 'size (acres)'
target2 = 'fire_spread (acres/day)'
target3 = 'duration'

In [4]:
# 3. Drop rows with missing values in predictors or targets
df = df.dropna(subset=predictors + [target1, target2, target3])


In [8]:
# 4. Feature matrix
X = df[predictors]
X = pd.get_dummies(df[predictors], drop_first=True).astype(float)


In [9]:
# 5. Define function to train and return a model
def train_rf_model(X, y, target_name):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model = RandomForestRegressor(
        n_estimators=200,
        max_depth=20,
        min_samples_split=2,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    score = r2_score(y_test, y_pred)
    print(f"R² score for {target_name}: {score:.3f}")
    return model

In [10]:
# 6. Train each model
model_size = train_rf_model(X, df[target1], target1)
model_spread = train_rf_model(X, df[target2], target2)
model_duration = train_rf_model(X, df[target3], target3)

R² score for size (acres): 0.587
R² score for fire_spread (acres/day): 0.433
R² score for duration: 0.539


In [11]:
# 7. Save each model to a separate file
output_dir = 'production_models'
os.makedirs(output_dir, exist_ok=True)

In [12]:
# save models
joblib.dump(model_size, f'{output_dir}/rf_model_size.pkl')
joblib.dump(model_spread, f'{output_dir}/rf_model_spread.pkl')
joblib.dump(model_duration, f'{output_dir}/rf_model_duration.pkl')
print("✅ Models saved:")
print(f"- {output_dir}/rf_model_size.pkl")
print(f"- {output_dir}/rf_model_spread.pkl")
print(f"- {output_dir}/rf_model_duration.pkl")

✅ Models saved:
- production_models/rf_model_size.pkl
- production_models/rf_model_spread.pkl
- production_models/rf_model_duration.pkl


In [ ]:
# example code to load in
# Define the path where models are saved
# model_dir = 'production_models'

# Load each model
# model_size = joblib.load(f'{model_dir}/rf_model_size.pkl')
# model_spread = joblib.load(f'{model_dir}/rf_model_spread.pkl')
# model_duration = joblib.load(f'{model_dir}/rf_model_duration.pkl')

# print("✅ Models loaded successfully.")

✅ Models loaded successfully.
